# CH5 - Exercise 5

In Chapter 4, we used logistic regression to predict the probability of
`default` using `income` and `balance` on the `Default` data set. We will
now estimate the test error of this logistic regression model using the
validation set approach. Do not forget to set a random seed before
beginning your analysis.

**(a)** Fit a logistic regression model that uses `income` and `balance` to
predict `default`.

**(b)** Using the validation set approach, estimate the test error of this
model. In order to do this, you must perform the following steps:

- i. Split the sample set into a training set and a validation set.
- ii. Fit a multiple logistic regression model using only the training observations.
- iii. Obtain a prediction of default status for each individual in the validation set by computing the posterior probability of default for that individual, and classifying the individual to the `default` category if the posterior probability is greater than 0.5.
- iv. Compute the validation set error, which is the fraction of the observations in the validation set that are misclassified.

**(c)** Repeat the process in (b) three times, using three different splits
of the observations into a training set and a validation set. Comment on the results obtained.

**(d)** Now consider a logistic regression model that predicts the probability
of `default` using `income`, `balance`, and a dummy variable for
`student`. Estimate the test error for this model using the validation
set approach. Comment on whether or not including a dummy variable
for `student` leads to a reduction in the test error rate.

In [1]:
import numpy as np
import pandas as pd
from matplotlib .pyplot import subplots
import statsmodels .api as sm
from ISLP import load_data
from ISLP.models import ( ModelSpec as MS ,
summarize )

from ISLP import confusion_table
from ISLP.models import contrast
from sklearn. discriminant_analysis import \
( LinearDiscriminantAnalysis as LDA ,
QuadraticDiscriminantAnalysis as QDA)
from sklearn. naive_bayes import GaussianNB
from sklearn. neighbors import KNeighborsClassifier
from sklearn. preprocessing import StandardScaler
from sklearn. model_selection import train_test_split
from sklearn. linear_model import LogisticRegression

from functools import partial
from sklearn.model_selection import (
    train_test_split, cross_validate, KFold, ShuffleSplit
)
from sklearn import clone 
from ISLP.models import sklearn_sm


In [2]:
Default = load_data("Default")
Default.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


In [3]:
def_train,def_valid = train_test_split(Default,
                                    test_size=int(Default.shape[0]/2),
                                    random_state=0)

### Split and fit on training 

In [4]:
design  = MS(['balance', 'income'])
X_train = design.fit_transform(def_train)
y_train = (def_train['default'] == 'Yes')

model   = sm.GLM(y_train, X_train, family=sm.families.Binomial())
results = model.fit()
summarize(results)


,coef,std err,z,P>|z|
intercept,-11.389600,0.635000,-17.935,0.000
balance,0.005600,0.000000,16.792,0.000
income,0.000016,0.000007,2.151,0.031


### Validation

In [5]:
X_valid = design.transform(def_valid)
y_valid = (def_valid['default'] == 'Yes')

pred = results.predict(X_valid)
pred = (pred > 0.5)

print('\n Validation set error: ',np.mean(pred != y_valid))


 Validation set error:  0.029


### Repeat for different splits


In [6]:
def evalError(
        terms,
        response,
        train,
        evalu):

    design = MS(terms)
    X_train = design.fit_transform(train)
    y_train = (train[response] == 'Yes')

    model = sm.GLM(y_train,X_train,family=sm.families.Binomial())
    results = model.fit()
    
    X_eval = design.transform(evalu)
    y_eval = (evalu[response] == 'Yes')

    pred = results.predict(X_eval)
    pred = (pred > 0.5)

    return np.mean(pred != y_eval)

length = 10
Errs_no_student = np.zeros(length)
for idx in range(length):
    def_train,def_valid = train_test_split(Default,
                                    test_size=int(Default.shape[0]/2),random_state=idx)
    
    Errs_no_student[idx] = evalError(terms=['balance', 'income'],
                        response='default',
                        train = def_train,
                        evalu = def_valid)
print(Errs_no_student)


[0.029  0.025  0.0248 0.0248 0.0238 0.0258 0.0268 0.0264 0.0232 0.0234]


### Introduction of a new predictor: **student**

In [7]:
Errs_student = np.zeros(length)
for idx in range(length):
    def_train,def_valid = train_test_split(Default,
                                    test_size=int(Default.shape[0]/2),random_state=idx)
    
    Errs_student[idx] = evalError(terms=['balance', 'income','student'],
                        response='default',
                        train = def_train,
                        evalu = def_valid)
print(Errs_student)



[0.0292 0.0262 0.0254 0.0252 0.0244 0.0256 0.028  0.0262 0.023  0.024 ]


In [8]:
print('Without student:', Errs_no_student)
print('With student:   ', Errs_student)
print('Difference:     ', Errs_no_student - Errs_student)

Without student: [0.029  0.025  0.0248 0.0248 0.0238 0.0258 0.0268 0.0264 0.0232 0.0234]
With student:    [0.0292 0.0262 0.0254 0.0252 0.0244 0.0256 0.028  0.0262 0.023  0.024 ]
Difference:      [-0.0002 -0.0012 -0.0006 -0.0004 -0.0006  0.0002 -0.0012  0.0002  0.0002
 -0.0006]


Including **student** does not improve the model in a perceptible way. 